# Export Enhanced Strategy Comparison to Multi-Sheet Excel

This notebook replicates the logic from `comparison_markowitz_ai_graph_enhanced.ipynb` and adds a dedicated section to export the backtest results (2025) into a multi-sheet Excel file. 

Each sheet represents one model: 
1. Markowitz Only
2. AI-Gated Markowitz
3. Full (AI + Graph + MK)
4. BTC Benchmark

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier

# --- 1. Load Data ---
file_path = '../dataset_2023_2025.xlsx'
if not os.path.exists(file_path):
    file_path = 'dataset_2023_2025.xlsx'

data = pd.read_excel(file_path, index_col=0, parse_dates=True)
returns = data.pct_change().dropna()
market_return = returns.mean(axis=1)

print(f"✓ Data Loaded: {len(data)} rows")

✓ Data Loaded: 990 rows


In [2]:
# --- 2. AI Gatekeeper Training ---
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
features['Mom_20'] = market_return.rolling(window=20).mean()
features['Mom_50'] = market_return.rolling(window=50).mean()
features['Target'] = (market_return.shift(-1) > 0).astype(int)
features = features.dropna()

train_mask = (features.index.year >= 2023) & (features.index.year <= 2024)
test_mask = (features.index.year == 2025)

X_train, y_train = features.loc[train_mask, ['Vol_20', 'Mom_20', 'Mom_50']], features.loc[train_mask, 'Target']
X_test, y_test = features.loc[test_mask, ['Vol_20', 'Mom_20', 'Mom_50']], features.loc[test_mask, 'Target']

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, min_samples_split=10)
rf_model.fit(X_train, y_train)
test_probs = pd.Series(rf_model.predict_proba(X_test)[:, 1], index=X_test.index)

print("✓ AI Gatekeeper Trained.")

✓ AI Gatekeeper Trained.


In [3]:
# --- 3. Strategy Functions ---

def get_mis_assets(returns_window, correlation_threshold=0.5):
    corr_mat = returns_window.corr()
    G = nx.Graph()
    assets = returns_window.columns
    G.add_nodes_from(assets)
    for i in range(len(assets)):
        for j in range(i+1, len(assets)):
            if corr_mat.iloc[i, j] > correlation_threshold:
                G.add_edge(assets[i], assets[j])
    mis = nx.approximation.maximum_independent_set(G)
    return list(mis)

def optimize_markowitz(selected_returns):
    if len(selected_returns.columns) == 0: return {}
    if len(selected_returns.columns) == 1: return {selected_returns.columns[0]: 1.0}
    
    mu = selected_returns.mean() * 252
    sigma = selected_returns.cov() * 252
    
    def neg_sharpe_ratio(weights):
        port_ret = np.sum(weights * mu)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(sigma, weights)))
        return -port_ret / port_vol if port_vol > 0 else 0
    
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(len(mu)))
    guess = [1. / len(mu)] * len(mu)
    
    try:
        res = minimize(neg_sharpe_ratio, guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return dict(zip(selected_returns.columns, res.x))
    except:
        return dict(zip(selected_returns.columns, guess))

def run_simulation(strategy_type, test_dates, returns, probs, lookback=30):
    current_value = 10000.0
    history = [current_value]
    daily_returns = [0.0]
    dates = [test_dates[0]]
    
    diag_regimes = []
    diag_n_assets = []
    diag_weights = []
    
    for i, date in enumerate(test_dates[:-1]):
        # 1. AI Decision
        is_bull = probs.loc[date] > 0.5 if strategy_type != 'Markowitz Only' else True
        diag_regimes.append('BULL' if is_bull else 'BEAR')
        
        loc_idx = returns.index.get_loc(date)
        window_rets = returns.iloc[loc_idx-lookback:loc_idx]
        
        if not is_bull:
            weights = {'CASH': 1.0}
        else:
            if strategy_type == 'Full':
                selected = get_mis_assets(window_rets)
                weights = optimize_markowitz(window_rets[selected])
            else:
                weights = optimize_markowitz(window_rets)
        
        diag_n_assets.append(len([w for w in weights if w != 'CASH']))
        diag_weights.append(weights)
        
        # 2. Daily PnL
        next_date = test_dates[i+1]
        next_rets = returns.loc[next_date]
        
        day_ret = 0 if 'CASH' in weights else sum(w * next_rets[a] for a, w in weights.items())
        
        current_value *= (1 + day_ret)
        history.append(current_value)
        daily_returns.append(day_ret)
        dates.append(next_date)
        
    # Match lengths for DataFrame
    diag = {
        'Regime': diag_regimes + [np.nan],
        'Active_Assets': diag_n_assets + [0],
        'Weights_Dict': diag_weights + [{}]
    }
    
    df_res = pd.DataFrame({
        'Date': dates,
        'Portfolio_Value': history,
        'Daily_Return': daily_returns,
        'Regime': diag['Regime'],
        'Active_Assets': diag['Active_Assets']
    }).set_index('Date')
    
    # Flatten Weights
    weights_df = pd.DataFrame(diag['Weights_Dict'], index=dates)
    df_res = pd.concat([df_res, weights_df], axis=1)
    
    return df_res

In [4]:
# --- 4. Run Backtests and Export ---
test_dates = X_test.index
output_file = "enhanced_portfolio_results_2025.xlsx"

print("Starting Backtests and Data Preparation...")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 1. Markowitz Only
    print("- Processing Markowitz Only...")
    df_m = run_simulation('Markowitz Only', test_dates, returns, test_probs)
    df_m.to_excel(writer, sheet_name='Markowitz Only')
    
    # 2. AI-Gated Markowitz
    print("- Processing AI-Gated Markowitz...")
    df_a = run_simulation('AI-Gated Markowitz', test_dates, returns, test_probs)
    df_a.to_excel(writer, sheet_name='AI-Gated Markowitz')
    
    # 3. Full Strategy
    print("- Processing Full (AI+Graph+MK)...")
    df_f = run_simulation('Full', test_dates, returns, test_probs)
    df_f.to_excel(writer, sheet_name='Full AI-Graph-MK')
    
    # 4. BTC Benchmark
    print("- Processing BTC Benchmark...")
    btc_price = data.loc[test_dates, 'BTC-USD']
    df_btc = pd.DataFrame({
        'Date': test_dates,
        'Portfolio_Value': (btc_price / btc_price.iloc[0] * 10000),
        'Daily_Return': btc_price.pct_change().fillna(0),
        'BTC_Weight': 1.0
    }).set_index('Date')
    df_btc.to_excel(writer, sheet_name='BTC Benchmark')

print(f"\n✅ SUCCESS: Results exported to {output_file}")

Starting Backtests and Data Preparation...
- Processing Markowitz Only...
- Processing AI-Gated Markowitz...
- Processing Full (AI+Graph+MK)...
- Processing BTC Benchmark...

✅ SUCCESS: Results exported to enhanced_portfolio_results_2025.xlsx
